# CD CosMx 6K resections: Treg raw counts

1. For each resection run, attach the AUCell labels (`04_resection_aucell_annotation.R`) and export the raw counts of all AUCell Tregs (`combined/raw_counts_tregs.csv`; cell IDs prefixed with patient and section).
2. Normalise, score Treg gene panels and keep strict Tregs (core Treg score ≥ 0, cell-cycle score < 10; 6,554 cells from 11 sections).
3. Export their raw counts (`combined/raw_counts_strict_tregs.csv`), the query of the Treg label transfer in `09_treg_label_transfer.R`.

In [ ]:
from pathlib import Path
import scanpy as sc
import pandas as pd
from scipy import sparse
import os
import gc

In [ ]:
sample_list =[   # run 18888_A5 is not used
'Cos1_6KDC_strict43563_SB_A1_091725BP6_09_10_2025_13_54_55_807',          
'Cos1_6KDC_strict43563_SB_A8_091725BP5_09_10_2025_13_41_29_437',       
'Cos1_6KDC_strict54033_B4_0715_11_08_2025_14_20_23_133',     
'Cos2_6KDC_strict18888_A10_0715_Alan_data_test_11_08_2025_14_25_11_455',  
'Cos2_6KDC_strict18888_A6_0715_Alan_Test_11_08_2025_14_23_09_489',  
'Cos3_6KDC_strict11808_A4_0716_11_08_2025_14_19_48_615',                  
'Cos3_6KDC_strict54033_B5_0716_11_08_2025_14_17_56_277',                
'Cos4_6KDC_strict11808_A4_0716_14_10_2025_12_34_38_727',
'Cos4_6KDC_strict54033_B7_0716_11_08_2025_14_18_41_738',
'Cos5_6KDC_strict43563_SB_A11_091725BP7_09_10_2025_13_42_42_775',         
'Cos5_6KDC_strict43563_SB_A4_091725BP8_09_10_2025_13_43_50_952'  
]
file_prefix = "/share/fsmresfiles/UC/AtoMx/CD_surgical_resection/"  # prefix to the h5ad file
file_suffix = '/Processed/'+'norm_log1p.h5ad' # suffix to the h5ad file
out_csv = "/share/fsmresfiles/UC/AtoMx/CD_surgical_resection/combined/raw_counts_tregs.csv"
cell_type_col = "cell_type_short"
treg_keyword = "Treg"

In [3]:
mapper= pd.read_csv('/share/fsmresfiles/UC/scRNA-seq/merged_cd/cell_type_category_map.csv')

In [4]:
# Clean the names
mapper['cell_type_aucell'] = mapper['cell type'].str.replace(r'[+\-\/()]', ' ', regex=True)
mapper['cell_type_aucell'] = mapper['cell_type_aucell'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Disambiguate duplicates by appending " 1", " 2", etc.
mapper['cell_type_aucell'] = (
    mapper.groupby('cell_type_aucell').cumcount().astype(str).replace('0', '', regex=False)
    .radd(mapper['cell_type_aucell'] + ' ').str.strip()
)

In [18]:
sample_list_short = []

In [ ]:
j = 0
for i in sample_list:
    df = pd.read_csv(file_prefix+i+'/aucell_anno.csv')
    df['label_clean'] = df['label'].str.rstrip()
    df=df.merge(mapper, how = 'left', left_on = 'label_clean',right_on = 'cell_type_aucell')
    if 'SB' in i:
        patient = i.split('strict')[1].split('_')[0]
        section = i.split('strict')[1].split('_')[2]
        comb = patient+'_'+section
    elif '11808' in i:
        patient = i.split('strict')[1].split('_')[0]
        section = i.split('strict')[1].split('_')[1]
        machine = i.split('_')[0]
        comb = patient+'_'+section+'_'+machine
    else:
        patient = i.split('strict')[1].split('_')[0]
        section = i.split('strict')[1].split('_')[1]
        comb = patient+'_'+section
    
    df['patient']=patient
    df['patient']=section
    df['patient_sec'] = comb
    df['sample']=i
    sample_list_short.append(comb)
    if j == 0:
        merged = df
    else:
        merged = pd.concat([merged,df],ignore_index=True)
    j+=1

In [30]:
def get_raw_counts_df(adata):
    """Return raw counts as DataFrame from layers['raw'] or .X"""
    mat = adata.layers['raw']
    if sparse.issparse(mat):
        mat = mat.toarray()
    df = pd.DataFrame(mat, index=adata.obs_names.astype(str), columns=adata.var_names.astype(str))
    return df

In [ ]:
def extract_tregs(h5ad_path, ann_df, cell_type_col="cell_type_short", treg_keyword="Treg", sample_name=None, sample_name_short=None):
    """Load h5ad, attach annotation, filter Tregs, return DataFrame of raw counts with new index"""
    adata = sc.read_h5ad(h5ad_path)
    
    # attach annotations
    merged_sub = ann_df[ann_df['sample']==sample_name]
    adata.obs[cell_type_col] = merged_sub['cell type short'].tolist()
        
    # filter Tregs
    mask = adata.obs[cell_type_col].astype(str).str.contains(treg_keyword)
    adata_tregs = adata[mask].copy()
    
    df_counts = get_raw_counts_df(adata_tregs)
    # new index = sample_cellid
    new_index = [f"{sample_name_short}_{cid}" for cid in df_counts.index.astype(str)]
    df_counts.index = new_index
    
    del adata
    gc.collect()
    return df_counts


In [32]:
all_dfs = []
for sample,sample_short in zip(sample_list,sample_list_short):
    h5ad_path = f"{file_prefix}{sample}{file_suffix}"
    df = extract_tregs(h5ad_path, merged, cell_type_col=cell_type_col, treg_keyword=treg_keyword, sample_name=sample, sample_name_short =sample_short )
    if df is not None:
        all_dfs.append(df)

Loading /share/fsmresfiles/UC/AtoMx/CD_surgical_resection/Cos1_6KDC_strict43563_SB_A1_091725BP6_09_10_2025_13_54_55_807/Processed/norm_log1p.h5ad ...
read in
554 Tregs found
  Extracted 554 Treg cells from Cos1_6KDC_strict43563_SB_A1_091725BP6_09_10_2025_13_54_55_807
Loading /share/fsmresfiles/UC/AtoMx/CD_surgical_resection/Cos1_6KDC_strict43563_SB_A8_091725BP5_09_10_2025_13_41_29_437/Processed/norm_log1p.h5ad ...
read in
1047 Tregs found
  Extracted 1047 Treg cells from Cos1_6KDC_strict43563_SB_A8_091725BP5_09_10_2025_13_41_29_437
Loading /share/fsmresfiles/UC/AtoMx/CD_surgical_resection/Cos1_6KDC_strict54033_B4_0715_11_08_2025_14_20_23_133/Processed/norm_log1p.h5ad ...
read in
4566 Tregs found
  Extracted 4566 Treg cells from Cos1_6KDC_strict54033_B4_0715_11_08_2025_14_20_23_133
Loading /share/fsmresfiles/UC/AtoMx/CD_surgical_resection/Cos2_6KDC_strict18888_A10_0715_Alan_data_test_11_08_2025_14_25_11_455/Processed/norm_log1p.h5ad ...
read in
615 Tregs found
  Extracted 615 Treg cells

In [33]:
combined = pd.concat(all_dfs, axis=0)


In [ ]:
os.makedirs(Path(out_csv).parent, exist_ok=True)
combined.to_csv(out_csv, index=True)

In [3]:
counts = pd.read_csv('/share/fsmresfiles/UC/AtoMx/CD_surgical_resection/combined/raw_counts_tregs.csv')

In [8]:
counts.index = counts['Unnamed: 0'].tolist()

In [9]:
counts=counts.drop('Unnamed: 0',axis = 1)

In [ ]:
# Convert to sparse matrix for efficiency
X_sparse = sparse.csr_matrix(counts.values)

# Create AnnData
adata = sc.AnnData(X=X_sparse)
adata.obs_names = counts.index.astype(str)   # cell IDs
adata.var_names = counts.columns.astype(str) # gene names

# Optional: store raw counts in layers
adata.layers["raw"] = X_sparse

In [14]:
sc.pp.normalize_total(adata, target_sum=1e4) 

In [28]:
panels = {
 'Activated_tissue': ['BATF','PRDM1','MAF','TNFRSF18','TNFRSF4'],
 'Tfr': ['BCL6','CXCR5','PDCD1','ICOS'],
 'Th17_like': ['RORC','CCR6','IL17A','IL17F'],
 'IFN_response': ['ISG15','IFI6','MX1','IFIT3'],
 'Cell_cycle': ['MKI67','TOP2A','STMN1'],
 'Treg_core':['FOXP3','IL2RA','CTLA4','TIGIT','ENTPD1','LRRC32','IL10']
}
for name, genes in panels.items():
    sc.tl.score_genes(adata, [g for g in genes if g in adata.var_names], score_name=name)

In [34]:
treg_strict = adata[adata.obs["Treg_core"] >= 0].copy()

In [36]:
treg_strict = treg_strict[treg_strict.obs["Cell_cycle"] < 10].copy()

In [39]:
raw_counts = pd.DataFrame(treg_strict.layers['raw'].toarray())
raw_counts.columns = treg_strict.var_names.tolist()
raw_counts.index = treg_strict.obs_names.tolist()

In [40]:
raw_counts.to_csv('/share/fsmresfiles/UC/AtoMx/CD_surgical_resection/combined/raw_counts_strict_tregs.csv')